# CS431 · E-BN pilot, Giai đoạn 0: đo độ lệch thống kê BatchNorm sau khi trộn (KHÁM PHÁ)

**Kết luận Mix A-core v1 giữ nguyên:** α\* = B-Clean, Mix A chưa thắng theo quy tắc đã khóa. Notebook này **không** sửa notebook v1 hay kết quả v1. Thiết kế: `docs/research/janek/design_bn_recalibration_pilot_janek.md`, kế hoạch §4.4.

**Giai đoạn 0 làm gì:**
- Model ở `eval()`, chỉ gắn hook đọc đầu vào của 2 lớp BN đang có tác dụng lúc inference: `det_head.binarize.1` và `.4`.
- **Không ghi gì** vào trọng số hay buffer; code kiểm SHA của toàn bộ `state_dict` trước và sau.
- Đo trên 1.000 nguồn **TRAIN** cố định × (Clean, LR2) = 2.000 ảnh, cho cả 5 checkpoint của v1. Hai endpoint là đối chứng.
- Không dùng ảnh A/B (remaining-validation) hay TEST để đo.

**Giai đoạn 1** (tính lại BN rồi chấm counts) **không có trong notebook này**. Nó chỉ mở khi nhóm đã thống nhất tiêu chí và Giai đoạn 0 có tín hiệu.

**Trạng thái:** code CPU có test ở máy local. **Phần GPU chưa chạy lần nào.** Mọi ngưỡng dừng hay đi tiếp ở đây là **đề xuất, Tuấn chưa duyệt**.

### Bạn cần chuẩn bị
1. Trên máy local, trong `cải tiến/mới`, chạy `python scripts/make_colab_upload_janek.py`. Tải `runtime/colab_upload/mixab_code.zip` lên `MyDrive/CS431_MixA/upload/`, ghi đè bản cũ.
2. **File TRAIN từ Tuấn**, đặt vào `MyDrive/CS431_MixA/bn_pilot/input/`, **chỉ một trong ba dạng**:
   - `manifest__train__textvqa_clean_degraded_same_textocr_word_gt.jsonl` (hoặc `.jsonl.gz`): bản đầy đủ. SHA-256 phải đúng `ca1b9163…c983d`, sai là dừng.
   - `train_ids.csv` với các cột `source_image_id,clean_relpath,clean_sha256` (thêm `clean_size` nếu có). Được chấp nhận, nhưng ghi rõ **chưa xác minh được hash manifest gốc**.
3. Runtime → GPU **A100**. Chạy Ô 1 → Ô 4, rồi Ô B1 → Ô B4.

Thiếu file TRAIN thì Ô B1 dừng và báo rõ, **trước** mọi bước xử lý ảnh hoặc dùng GPU. Ô 1–4 vẫn chạy được để dựng sẵn môi trường.

### Dữ liệu tạm và dữ liệu giữ lại
- Tạm (`/content/cs431_tmp/`): ảnh TRAIN đã giải nén, ảnh LR2 TRAIN, code.
- Giữ lại trên Drive:
  - `bn_pilot/input/` (file TRAIN của bạn);
  - `bn_pilot/train_check/` (kết quả kiểm, subset, SHA ảnh);
  - `cache/textvqa_train_bn1000_*.zip` (1.000 ảnh Clean TRAIN);
  - `runs/bn_stage0_v1_<GPU>/` (kết quả, resume được);
  - `logs/`, `export/`.

## Ô 1 · Cấu hình và mount Drive (chạy lại ô này sau mỗi lần restart)

Chỉ cần sửa `DRIVE_ROOT` nếu bạn đặt thư mục ở chỗ khác. Các commit được **ghim cố định** để kết quả so được với của Tuấn:
- MONKEY_V2 `dc19b85` là commit có bundle `pr_quad_audit_v1`.
- MonkeyOCRv2 `da96afe` là bản đã đối chiếu ở máy local.

Đổi commit thì đổi luôn identity của lần chạy, nên phải ghi lý do vào WORKLOG.

In [ ]:
# ===================== Ô 1 · CẤU HÌNH =====================
import json, os, subprocess, sys, zipfile
from pathlib import Path

DRIVE_ROOT = '/content/drive/MyDrive/CS431_MixA'      # <-- thư mục của bạn trên Google Drive

PIN = dict(
    MONKEY_V2_URL='https://github.com/ncat218/MONKEY_V2.git',
    MONKEY_V2_COMMIT='dc19b85c8f73d56dd0510165069597363fe25380',
    MONKEYOCRV2_URL='https://github.com/Yuliang-Liu/MonkeyOCRv2.git',
    MONKEYOCRV2_COMMIT='da96afede92f198c19aca74e1e5f5aa7f4c26af2',
    TEXTVQA_ZIP_URL='https://dl.fbaipublicfiles.com/textvqa/images/train_val_images.zip',
    TEXTVQA_ZIP_BYTES=7072297970,                          # đo bằng HTTP HEAD ngày 20/11/2026
    ENCODER_REPO='zenosai/MonkeyOCRv2-AS',
    BCLEAN_SHA256='94f0e4ecbb6881b4430c841845994c6568c145af7bb54fc41514eb0c78f3a6aa',
    B1_SHA256='9c3814e494a73a0a6089ba5a6585f079ea8f5a6defdc9347213bb98a64337147',
    # Môi trường Tuấn đã kiểm (MONKEY_V2 docs/handoff/checkpoints.md + MonkeyOCRv2 detection/mmocr/README_janek.md)
    TORCH='2.9.0', TORCHVISION='0.24.0', TORCHAUDIO='2.9.0', TORCH_INDEX='https://download.pytorch.org/whl/cu128',
    MMENGINE='0.10.7', MMCV='2.0.1', MMOCR='1.0.1', PILLOW='11.3.0', OPENCV='4.10.0.84',
)
PROBE_SOURCES = 40       # probe cố định 40 nguồn x 2 điều kiện; KHÔNG đổi giữa các lần resume
PROBE_CHUNK = 16         # lưu tiến độ sau mỗi 16 task
FULL_CHUNK = 64          # chạy đầy đủ: lưu tiến độ sau mỗi 64 task (~20 giây trên A100)
REFERENCE_POLICY = 'same_gpu_endpoints'   # kế hoạch §4.3, phương án (a); KHÔNG đổi nếu chưa ghi quyết định mới

# ---- mount Drive (chỉ chạy trên Colab) ----
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    raise SystemExit('Notebook này chạy trên Google Colab (cần google.colab để mount Drive).')

D = Path(DRIVE_ROOT)
UPLOAD, CACHE, WEIGHTS = D/'upload', D/'cache', D/'weights/merge_weights_v1'
RUNS, REPORTS, LOGS, EXPORT = D/'runs', D/'reports', D/'logs', D/'export'
T = Path('/content/cs431_tmp')                       # TẠM: mất khi VM reset
CODE, MV2, MOCR = T/'code', T/'MONKEY_V2', T/'MonkeyOCRv2'
IMAGES, PREPARED, DOWNLOADS = T/'textvqa', T/'prepared', T/'downloads'
CONFIG_PATH = T/'inference_config_colab.py'
for p in (UPLOAD, CACHE, RUNS, REPORTS, LOGS, EXPORT, T):
    p.mkdir(parents=True, exist_ok=True)

# ---- code của mình (mixab) từ Drive -> /content (giải nén lại khi zip đổi) ----
CODE_ZIP = UPLOAD/'mixab_code.zip'
if not CODE_ZIP.is_file():
    raise SystemExit(f'Chưa thấy {CODE_ZIP}. Trên máy local chạy `python scripts/make_colab_upload_janek.py` '
                     'rồi tải runtime/colab_upload/mixab_code.zip lên thư mục upload/ trên Drive.')
import hashlib
code_sha = hashlib.sha256(CODE_ZIP.read_bytes()).hexdigest()
marker = CODE/'.code_zip_sha256'
if not (marker.is_file() and marker.read_text() == code_sha):
    if CODE.exists():
        CODE.rename(T/f'code_old_{code_sha[:8]}_{int(__import__("time").time())}')
    with zipfile.ZipFile(CODE_ZIP) as zf:
        zf.extractall(CODE)
    marker.write_text(code_sha)
if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))
import importlib
import mixab.colab_helpers_janek as ch
importlib.reload(ch)

# ---- GPU ----
gpu_name = (ch.output_of(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader']) or '').splitlines()
if not gpu_name:
    raise SystemExit('Không thấy GPU. Runtime -> Change runtime type -> chọn T4 hoặc A100, rồi chạy lại Ô 1.')
GPU = ch.gpu_label(gpu_name[0])
if not ch.gpu_supported(gpu_name[0]):
    print(f'CẢNH BÁO: GPU {gpu_name[0]} không phải T4/A100 (ngoài phạm vi đã thống nhất). Nên đổi runtime.')
PY = sys.executable
ENV = dict(MONKEY_V2_ROOT=str(MV2), PYTHONPATH=str(CODE))
BUNDLE = MV2/'docs/handoff/artifacts/pr_quad_audit_v1'
print(f'GPU: {gpu_name[0]} -> nhãn {GPU}\nPython: {sys.version.split()[0]}\nCode zip SHA-256: {code_sha[:16]}...\n'
      f'Drive (giữ lại): {D}\nTạm: {T}')

## Ô 2 · Kiểm GPU và cài môi trường (E1)

Thứ tự cài theo hướng dẫn chính thức của MonkeyOCRv2 (`detection/mmocr/README_janek.md`):
1. torch 2.9.0 cu128
2. mmengine 0.10.7
3. **MMCV 2.0.1 build từ source** (không có wheel cho torch 2.9)
4. `install.sh`: clone MMOCR v1.0.1, chép add-on có `MonkeyOCRv2ViTAEBackbone`, cài mmdet 3.1.0, transformers 4.57.1, và vá imgaug/mmengine
5. ghim lại Pillow 11.3.0 và OpenCV 4.10.0.84

- **Build MMCV lâu**, có thể vài chục phút với 2 CPU của T4 (chưa đo). Wheel được lưu vào `cache/wheels/` trên Drive với `TORCH_CUDA_ARCH_LIST=7.5;8.0`, nên lần sau dùng chung cho cả T4 và A100, miễn là cùng phiên bản Python.
- Nếu Colab mặc định dùng `nvcc` 13.0, ô này tìm CUDA toolkit **12.8** riêng; nếu thiếu thì cài `cuda-toolkit-12-8` từ NVIDIA trên Ubuntu 22.04/24.04 x86_64. Không cài gói driver. Bước này cần tải thêm dữ liệu và chỉ còn trên VM hiện tại.
- Python 3.13 vẫn chưa được kiểm với toàn bộ stack này; sửa toolkit không đồng nghĩa đã xác nhận tương thích Python. Chỉ coi E1 đạt sau kiểm import và doctor.
- Mọi lệnh đều được ghi log vào `logs/install_<GPU>.log`. Ô có thể chạy lại: phần nào đúng phiên bản rồi thì bỏ qua.
- **Restart:** các bước nặng đều chạy trong tiến trình con, nên thường **không cần restart**. Nếu Colab hiện nút *Restart session* sau khi pip đổi gói, bấm restart, **chạy lại Ô 1**, rồi chạy tiếp **Ô 2** (bỏ qua rất nhanh) và **Ô 3**.

In [ ]:
# ===================== Ô 2 · CÀI MÔI TRƯỜNG =====================
import platform, re, shutil
LOG = LOGS/f'install_{GPU}.log'
pyver = sys.version_info
if not ((3, 10) <= pyver[:2] <= (3, 12)):
    print(f'CẢNH BÁO: Python {platform.python_version()} chưa được kiểm với MMCV 2.0.1 (README dùng 3.11).')

def ver(name):
    return ch.package_version(name)

def pip(*args):
    ch.run([PY, '-m', 'pip', 'install', '-q', *args], LOG, title='pip install ' + ' '.join(args))

# 1) torch
if ver('torch') != f"{PIN['TORCH']}+cu128" or ver('torchvision') != f"{PIN['TORCHVISION']}+cu128":
    pip(f"torch=={PIN['TORCH']}", f"torchvision=={PIN['TORCHVISION']}", '--index-url', PIN['TORCH_INDEX'])
# Colab may retain torchaudio built for its original torch. Transformers can import it indirectly.
if ver('torchaudio') != f"{PIN['TORCHAUDIO']}+cu128":
    pip('--no-deps', f"torchaudio=={PIN['TORCHAUDIO']}+cu128", '--index-url', PIN['TORCH_INDEX'])
cuda_ok = ch.output_of([PY, '-c', 'import torch;print(torch.version.cuda, torch.cuda.is_available(), torch.cuda.get_arch_list())'])
print('torch CUDA:', cuda_ok)
if not cuda_ok or 'True' not in cuda_ok:
    ch.fail('torch không dùng được CUDA. Kiểm runtime GPU rồi chạy lại Ô 1 và Ô 2.')

# 2) mmengine + công cụ build
if ver('mmengine') != PIN['MMENGINE']:
    pip(f"mmengine=={PIN['MMENGINE']}")
pip('setuptools<81', 'wheel', 'ninja')

# 3) MMCV 2.0.1 với ops (dùng wheel cache trên Drive nếu có)
def mmcv_ok():
    if ver('mmcv') != PIN['MMCV']:
        return False
    probe = subprocess.run([PY, '-c', 'from mmcv.ops import nms; print("IMPORT_NMS_OK")'],
                           capture_output=True, text=True)
    with LOG.open('a', encoding='utf-8') as log:
        log.write('\nMMCV import probe:\n' + probe.stdout + probe.stderr)
    if probe.returncode != 0:
        print(probe.stdout + probe.stderr)
    return probe.returncode == 0 and 'IMPORT_NMS_OK' in probe.stdout.splitlines()

wheel_dir = CACHE/f"wheels/mmcv{PIN['MMCV']}_py{pyver[0]}{pyver[1]}_torch{PIN['TORCH']}_cu128_sm75-80"
if not mmcv_ok():
    cached = sorted(wheel_dir.glob('mmcv-*.whl'))
    if cached:
        print('Cài MMCV từ wheel cache:', cached[-1].name)
        pip('--no-deps', str(cached[-1]))
    else:
        nvcc = shutil.which('nvcc')
        candidates = ['/usr/local/cuda-12.8', os.environ.get('CUDA_HOME'),
                      str(Path(nvcc).resolve().parents[1]) if nvcc else None,
                      '/usr/local/cuda']
        cuda_home = ch.find_cuda_toolkit(candidates)
        if not cuda_home:
            distro = platform.freedesktop_os_release()
            if (distro.get('ID') != 'ubuntu' or distro.get('VERSION_ID') not in ('22.04', '24.04')
                    or platform.machine() != 'x86_64'):
                ch.fail('Chưa có toolkit 12.8; tự cài chỉ hỗ trợ Ubuntu 22.04/24.04 x86_64. '
                        f'Môi trường hiện tại: {distro}, {platform.machine()}')
            print('Thiếu CUDA toolkit 12.8. Cài toolkit riêng từ NVIDIA (không cài driver)...')
            from urllib.request import urlretrieve
            repo = 'ubuntu' + distro['VERSION_ID'].replace('.', '')
            keyring = T/'cuda-keyring_1.1-1_all.deb'
            urlretrieve(f'https://developer.download.nvidia.com/compute/cuda/repos/{repo}/x86_64/'
                        'cuda-keyring_1.1-1_all.deb', keyring)
            ch.run(['dpkg', '-i', keyring], LOG)
            ch.run(['apt-get', 'update'], LOG)
            ch.run(['apt-get', 'install', '-y', '--no-install-recommends', 'cuda-toolkit-12-8'], LOG,
                   env={'DEBIAN_FRONTEND': 'noninteractive'})
            cuda_home = ch.find_cuda_toolkit(['/usr/local/cuda-12.8'])
            if not cuda_home:
                ch.fail('Đã cài nhưng chưa tìm thấy nvcc 12.8 và cuda.h; xem log cài đặt.')
        src = T/'mmcv_src'
        if not (src/'setup.py').is_file():
            ch.run(['git', 'clone', '-b', f"v{PIN['MMCV']}", '--depth', '1', 'https://github.com/open-mmlab/mmcv.git', src], LOG)
        setup = (src/'setup.py').read_text()
        setup = setup.replace('-std=c++14', '-std=c++17')  # README chính thức
        # Python 3.13+: exec/locals need an explicit namespace (PEP 667).
        old_version = "        exec(compile(f.read(), version_file, 'exec'))\n    return locals()['__version__']"
        new_version = "        namespace = {}\n        exec(compile(f.read(), version_file, 'exec'), namespace)\n    return namespace['__version__']"
        if old_version in setup:
            setup = setup.replace(old_version, new_version)
        elif new_version not in setup:
            ch.fail('MMCV setup.py khác cấu trúc get_version đã kiểm; dừng để kiểm tra.')
        (src/'setup.py').write_text(setup)
        print(f'Build MMCV (CUDA_HOME={cuda_home}, nvcc 12.8, {os.cpu_count()} CPU). Có thể mất lâu...')
        build_env = dict(CUDA_HOME=cuda_home, MMCV_WITH_OPS='1', MAX_JOBS=str(os.cpu_count() or 2),
                         PATH=f"{cuda_home}/bin:{os.environ.get('PATH', '')}",
                         TORCH_CUDA_ARCH_LIST='7.5;8.0')
        wheel_tmp = T/'wheelhouse'
        ch.run([PY, '-m', 'pip', 'wheel', '-v', '--no-build-isolation', '--no-deps', '-w', wheel_tmp, src], LOG,
               env=build_env, title='build MMCV wheel')
        built = sorted(wheel_tmp.glob('mmcv-*.whl'))
        if not built:
            ch.fail('Build MMCV không tạo ra wheel; xem log.')
        wheel_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(built[-1], wheel_dir/built[-1].name)
        pip('--no-deps', str(built[-1]))
    if not mmcv_ok():
        ch.fail('MMCV vẫn chưa import được mmcv.ops.nms sau khi cài; xem log.')

# 4) MonkeyOCRv2 add-on + MMOCR v1.0.1 (install.sh chính thức)
if not (MOCR/'.git').is_dir():
    ch.run(['git', 'clone', PIN['MONKEYOCRV2_URL'], MOCR], LOG)
ch.run(['git', '-C', MOCR, 'checkout', '--detach', PIN['MONKEYOCRV2_COMMIT']], LOG)
addon = MOCR/'detection/mmocr'
mmocr_root = addon/'mmocr'
backbone_file = mmocr_root/'mmocr/models/common/backbones/monkeyocr_v2_vitae.py'
mmocr_loc = ch.output_of([PY, '-c', 'import mmocr,os;print(os.path.dirname(mmocr.__file__))'])
if not (ver('mmocr') == PIN['MMOCR'] and backbone_file.is_file() and mmocr_loc and str(mmocr_root) in mmocr_loc):
    # Prepare upstream checkout before install.sh copies add-ons and installs it.
    # MMOCR 1.0.1 has the same implicit exec/locals issue as MMCV on Python 3.13.
    if not (mmocr_root/'.git').is_dir():
        ch.run(['git', 'clone', '--branch', 'v1.0.1', '--depth', '1',
                'https://github.com/open-mmlab/mmocr.git', mmocr_root], LOG)
    setup_path = mmocr_root/'setup.py'
    setup = setup_path.read_text()
    old_exec = "        exec(compile(f.read(), version_file, 'exec'))"
    new_exec = "        namespace = {}\n        exec(compile(f.read(), version_file, 'exec'), namespace)"
    if old_exec in setup:
        if setup.count(old_exec) != 1:
            ch.fail('MMOCR setup.py có nhiều exec khác dự kiến; cần kiểm tra.')
        setup = setup.replace(old_exec, new_exec)
        setup = setup.replace("locals()['short_version']", "namespace['short_version']")
        setup = setup.replace("locals()['__version__']", "namespace['__version__']")
        setup_path.write_text(setup)
    elif new_exec not in setup:
        ch.fail('MMOCR setup.py khác cấu trúc get_version đã kiểm; cần kiểm tra.')
    # install.sh gọi `python`/`pip` theo PATH: đặt interpreter của kernel lên đầu PATH
    ch.run(['bash', 'install.sh'], LOG, cwd=addon, title='MonkeyOCRv2 detection/mmocr/install.sh',
           env=dict(PATH=f"{Path(PY).parent}:{os.environ.get('PATH', '')}", PIP_VERBOSE='1'))

# 5) ghim lại Pillow / OpenCV (requirements của MMOCR có thể đã kéo bản khác)
if ver('Pillow') != PIN['PILLOW']:
    pip(f"Pillow=={PIN['PILLOW']}")
if ver('opencv-python') != PIN['OPENCV'] or any(ver(n) for n in ('opencv-python-headless', 'opencv-contrib-python')):
    ch.run([PY, '-m', 'pip', 'uninstall', '-y', 'opencv-python-headless', 'opencv-contrib-python',
            'opencv-contrib-python-headless'], LOG)
    # gỡ một biến thể có thể làm hỏng thư mục cv2 dùng chung -> cài lại opencv-python sạch
    pip('--force-reinstall', '--no-deps', f"opencv-python=={PIN['OPENCV']}")

# 6) kiểm cuối trong tiến trình MỚI (không dùng module cũ của kernel)
VERIFY = """
import hashlib, importlib.metadata as m, importlib.util, json, torch
from mmocr.utils import register_all_modules
from mmocr.registry import MODELS
from mmcv.ops import nms
register_all_modules(init_default_scope=True)
spec = importlib.util.find_spec('mmocr.models.textdet.postprocessors.db_postprocessor')
out = dict(versions={k: m.version(k) for k in ('torch','torchvision','torchaudio','mmengine','mmcv','mmdet','mmocr','Pillow','opencv-python','numpy','transformers')},
           cuda=torch.cuda.is_available(), device=torch.cuda.get_device_name(0), capability=list(torch.cuda.get_device_capability(0)),
           backbone_registered=MODELS.get('MonkeyOCRv2ViTAEBackbone') is not None,
           db_postprocessor_sha256=hashlib.sha256(open(spec.origin,'rb').read()).hexdigest(), db_postprocessor_path=spec.origin)
print(json.dumps(out))
"""
raw = ch.output_of([PY, '-c', VERIFY])
if not raw:
    ch.fail('Kiểm cuối môi trường thất bại khi import (torch/mmcv/mmocr). Chạy lại Ô 2 và xem log: ' + str(LOG))
env_report = json.loads(raw.splitlines()[-1])
env_path = ch.new_path(REPORTS, f'env_{GPU}', '.json')
env_path.write_text(json.dumps(dict(env_report, pins=PIN), indent=1))
print(json.dumps(env_report, indent=1))
problems = []
if env_report['db_postprocessor_sha256'] != '9bfff8aaaca103dd08d7565f2b7a5410ab20a6427fe1f0baad5a5d04d207c8b8':
    problems.append('DBPostprocessor khác bản Tuấn đã ghim')
if not env_report['backbone_registered']:
    problems.append('MonkeyOCRv2ViTAEBackbone chưa được đăng ký')
if env_report['versions']['Pillow'] != PIN['PILLOW']:
    problems.append('Pillow không phải 11.3.0')
if problems:
    ch.fail('Môi trường chưa đạt: ' + '; '.join(problems))
print(f'\nE1 (phần cài đặt) xong. Report: {env_path}\nĐây CHƯA phải doctor; chạy tiếp Ô 3 rồi Ô 4.')

## Ô 3 · Dữ liệu và weights

Thứ tự phụ thuộc, mỗi bước bỏ qua nếu đã đúng SHA:
1. **MONKEY_V2** ở commit ghim. Tải 2 file weights và các artifact LFS trong bundle (gồm `review_cards.zip`), rồi chạy `verify_bundle.py --counts --weights` (phải PASS).
2. **Test CPU** của `mixab` trên đúng bundle và weights này.
3. **Encoder** MonkeyOCRv2-AS: tải vào `cache/` trên Drive và ghi lại revision. Sau đó tạo bản config chỉ đổi **2 đường dẫn encoder**, mọi thứ khác giữ nguyên.
4. **Ảnh Clean:** lấy từ `cache/textvqa_val2262_clean.zip` trên Drive nếu đã có. Nếu chưa, tải zip TextVQA 7,07 GB vào thư mục tạm, chỉ trích 2.262 ảnh, kiểm SHA từng ảnh với `source_cohort.csv`, rồi lưu bản cache lên Drive. Lần sau không phải tải lại.
5. **5 checkpoint merge** trên Drive: dùng lại nếu SHA khớp manifest. Nếu có thư mục dở dang thì đổi tên chứ không xóa, rồi tạo lại.
6. **Ảnh native LR2** trong thư mục tạm: `prepare-lr2` dùng đúng hàm `native_lr2` của Tuấn và kiểm SHA từng ảnh.

In [ ]:
# ===================== Ô 3 · DỮ LIỆU + WEIGHTS =====================
LOG = LOGS/f'data_{GPU}.log'
# 3.1 MONKEY_V2 (tham chiếu, không sửa)
if not (MV2/'.git').is_dir():
    ch.run(['git', 'clone', PIN['MONKEY_V2_URL'], MV2], LOG, env=dict(GIT_LFS_SKIP_SMUDGE='1'))
ch.run(['git', '-C', MV2, 'checkout', '--detach', PIN['MONKEY_V2_COMMIT']], LOG, env=dict(GIT_LFS_SKIP_SMUDGE='1'))
if ch.output_of(['git', 'lfs', 'version']) is None:
    ch.run(['bash', '-c', 'apt-get -qq update && apt-get -qq install -y git-lfs'], LOG, title='install git-lfs')
ch.run(['git', '-C', MV2, 'lfs', 'install', '--local'], LOG)
weights_rel = ['docs/handoff/weights/bclean-inference.pth', 'docs/handoff/weights/b1-inference.pth']
lfs_include = weights_rel + ['docs/handoff/artifacts/pr_quad_audit_v1/**']
ch.run(['git', '-C', MV2, 'lfs', 'pull', '--include=' + ','.join(lfs_include), '--exclude='], LOG)
for rel, want in zip(weights_rel, (PIN['BCLEAN_SHA256'], PIN['B1_SHA256'])):
    if ch.sha256(MV2/rel) != want:
        ch.fail(f'{rel} sai SHA-256 (Git LFS chưa tải đủ?).')
dirty = ch.output_of(['git', '-C', MV2, 'status', '--porcelain', '--untracked-files=no'])
if dirty:
    ch.fail(f'Checkout MONKEY_V2 bị sửa (dirty), không dùng làm tham chiếu được:\n{dirty}')
ch.run([PY, BUNDLE/'verify_bundle.py', '--counts', '--weights'], LOG, cwd=MV2, title='verify_bundle.py --counts --weights')

# 3.2 test CPU của mixab trên đúng bundle/weights này
ch.run([PY, '-m', 'unittest', 'discover', '-s', 'tests', '-t', '.'], LOG, cwd=CODE, env=ENV, title='mixab unit tests')

# 3.3 encoder + config
ENCODER = CACHE/'encoder_monkeyocrv2_as'
if not ((ENCODER/'model.safetensors').is_file() and (ENCODER/'config.json').is_file()):
    ch.run([PY, '-c', f"from huggingface_hub import snapshot_download; snapshot_download({PIN['ENCODER_REPO']!r}, local_dir={str(ENCODER)!r})"],
           LOG, title='download encoder')
    rev = ch.output_of([PY, '-c', f"from huggingface_hub import HfApi; print(HfApi().model_info({PIN['ENCODER_REPO']!r}).sha)"])
    (ENCODER/'_revision.txt').write_text(f'{rev}\n')
print('Encoder revision:', (ENCODER/'_revision.txt').read_text().strip() if (ENCODER/'_revision.txt').is_file() else 'chưa ghi')
patched = ch.patch_inference_config((MV2/'docs/handoff/inference_config.py').read_text(encoding='utf-8'), ENCODER)
if not CONFIG_PATH.is_file() or CONFIG_PATH.read_text(encoding='utf-8') != patched:
    CONFIG_PATH.write_text(patched, encoding='utf-8')

# 3.4 ảnh Clean (2.262 ảnh)
items = ch.needed_clean_images(BUNDLE)
names = [r for r, _ in items]
CACHE_ZIP = CACHE/'textvqa_val2262_clean.zip'
if CACHE_ZIP.is_file():
    ch.extract_members(CACHE_ZIP, names, IMAGES)
else:
    big = DOWNLOADS/'train_val_images.zip'
    DOWNLOADS.mkdir(parents=True, exist_ok=True)
    if not (big.is_file() and big.stat().st_size == PIN['TEXTVQA_ZIP_BYTES']):
        ch.run(['wget', '-c', '-q', '--show-progress', '-O', big, PIN['TEXTVQA_ZIP_URL']], LOG, title='download TextVQA zip (7 GB)')
    if big.stat().st_size != PIN['TEXTVQA_ZIP_BYTES']:
        ch.fail(f'Zip TextVQA có {big.stat().st_size} byte, kỳ vọng {PIN["TEXTVQA_ZIP_BYTES"]}; chạy lại để tải tiếp.')
    ch.extract_members(big, names, IMAGES)
check = ch.verify_images(IMAGES, items)
print('Ảnh Clean:', check)
if check['ok'] != len(items):
    ch.fail(f'Ảnh Clean chưa khớp SHA: {check}')
if not CACHE_ZIP.is_file():
    ch.build_cache_zip(IMAGES, names, CACHE_ZIP)
    print('Đã lưu cache ảnh lên Drive:', CACHE_ZIP)
    big.unlink()                                              # file tải 7 GB chỉ nằm ở thư mục tạm
print('Ảnh Clean OK: 2.262/2.262 SHA khớp')

# 3.5 5 checkpoint merge (Drive)
manifest = WEIGHTS/'merge_manifest.json'
def weights_ok():
    if not manifest.is_file():
        return False
    m = json.loads(manifest.read_text())
    good = all((WEIGHTS/Path(r['checkpoint']).name).is_file() and ch.sha256(WEIGHTS/Path(r['checkpoint']).name) == r['checkpoint_sha256']
               for r in m['models'])
    return good and m['endpoints']['bclean']['sha256'] == PIN['BCLEAN_SHA256'] and m['endpoints']['b1']['sha256'] == PIN['B1_SHA256']
if weights_ok():
    print('Dùng lại 5 checkpoint merge đã có trên Drive (SHA khớp).')
else:
    moved = ch.move_aside(WEIGHTS)
    if moved:
        print('Thư mục weights cũ chưa đầy đủ đã được đổi tên thành:', moved)
    ch.run([PY, '-m', 'mixab.weight_merge_janek', '--bclean', MV2/weights_rel[0], '--b1', MV2/weights_rel[1], '--output', WEIGHTS],
           LOG, cwd=CODE, env=ENV, title='mixab.weight_merge_janek')
    if not weights_ok():
        ch.fail('Checkpoint merge vừa tạo không qua kiểm SHA.')

# 3.6 native LR2 (tạm, tái tạo được, kiểm SHA từng ảnh)
ch.run([PY, '-m', 'mixab.merge_audit_janek', 'prepare-lr2', '--bundle', BUNDLE, '--clean-root', IMAGES, '--prepared-root', PREPARED],
       LOG, cwd=CODE, env=ENV, title='prepare-lr2')
print('\nÔ 3 xong: bundle PASS, test PASS, encoder + config, 2.262 ảnh Clean, 5 checkpoint, LR2.')

## Ô 4 · Doctor (cổng vào của probe)

Doctor ghi report mới với timestamp vào `reports/doctor_<GPU>_*.json`, không bao giờ ghi đè report cũ.
- **Chỉ khi 0 FAIL** thì ô này mới ghi "giấy thông hành" cho phiên hiện tại (`/content/cs431_tmp/doctor_pass.json`).
- Không có tùy chọn bỏ qua hay nới kiểm tra. Còn FAIL thì sửa đúng mục đó rồi chạy lại.
- WARN (ví dụ T4 không có TF32, phiên bản OpenCV hay torch lệch nhỏ) **không chặn**, nhưng được ghi lại để đối chiếu nếu cổng 3a lệch.

In [ ]:
# ===================== Ô 4 · DOCTOR =====================
doctor_path = ch.new_path(REPORTS, f'doctor_{GPU}', '.json')
LOG = LOGS/f'doctor_{GPU}.log'
cmd = [PY, '-m', 'mixab.doctor_janek', '--bundle', BUNDLE, '--clean-root', IMAGES, '--prepared-root', PREPARED,
       '--model-config', CONFIG_PATH, '--weights-dir', WEIGHTS, '--output', doctor_path]
try:
    ch.run(cmd, LOG, cwd=CODE, env=ENV, title='mixab.doctor_janek')
except ch.StepError:
    pass                                   # doctor exit 1 khi có FAIL; đọc report bên dưới
if not doctor_path.is_file():
    ch.fail(f'Doctor không tạo được report; xem {LOG}')
report = json.loads(doctor_path.read_text())
fails = ch.failed_checks(report)
pass_flag = T/'doctor_pass.json'
if pass_flag.exists():
    pass_flag.unlink()                     # giấy thông hành chỉ hợp lệ cho lần doctor mới nhất
if fails:
    ch.fail('Doctor còn FAIL, chưa được chạy probe:\n' + '\n'.join(f"- {r['check']}: {r['detail']}" for r in fails)
            + f'\nReport: {doctor_path}')
pass_flag.write_text(json.dumps(dict(report=str(doctor_path), gpu=GPU, code_zip_sha256=code_sha)))
print(f'\nDoctor: 0 FAIL. Report: {doctor_path}\nĐược phép chạy các ô BN (B1 → B4).')

## Ô B1 · Kiểm file TRAIN và chọn subset cố định (chỉ CPU, không cần ảnh)

- Đọc file trong `bn_pilot/input/`. Manifest đầy đủ phải đúng SHA Tuấn xác nhận; danh sách ID thì ghi `NOT_VERIFIABLE_ID_LIST`.
- Kiểm: đúng **15.222** nguồn, không trùng ID, SHA hợp lệ, đường dẫn nằm trong `train_images/`.
- Kiểm **không trùng** ID hoặc SHA ảnh Clean với 2.262 nguồn remaining-validation đã công bố. 1.000 nguồn S1/S2 của validation chưa công bố nên không kiểm được; kết quả ghi rõ điều này.
- Chọn 1.000 nguồn cố định: sắp theo ID, `numpy.default_rng(20261120)`. Lần chạy sau phải ra đúng subset cũ, nếu khác thì dừng, không ghi đè.

Không tự chia lại TextVQA. Không có file của Tuấn thì dừng ở đây.

In [ ]:
# ===================== Ô B1 · FILE TRAIN + SUBSET =====================
BN = D/'bn_pilot'
BN_INPUT, BN_CHECK = BN/'input', BN/'train_check'
for p in (BN_INPUT, BN_CHECK):
    p.mkdir(parents=True, exist_ok=True)
TRAIN_NAMES = ['manifest__train__textvqa_clean_degraded_same_textocr_word_gt.jsonl',
               'manifest__train__textvqa_clean_degraded_same_textocr_word_gt.jsonl.gz', 'train_ids.csv']
found = [BN_INPUT/n for n in TRAIN_NAMES if (BN_INPUT/n).is_file()]
if not found:
    ch.fail('Chưa có file TRAIN. Dừng trước mọi bước xử lý ảnh/GPU.\n'
            f'Đặt MỘT trong các file sau vào {BN_INPUT}:\n  - ' + '\n  - '.join(TRAIN_NAMES) +
            '\n(manifest đầy đủ của Tuấn, SHA-256 ca1b91636cbe…c983d; hoặc train_ids.csv có cột '
            'source_image_id,clean_relpath,clean_sha256) rồi chạy lại Ô B1.')
if len(found) > 1:
    ch.fail(f'Có nhiều file TRAIN trong {BN_INPUT}: {[p.name for p in found]}. Chỉ giữ một file.')
LOG = LOGS/f'bn_check_{GPU}.log'
ch.run([PY, '-m', 'mixab.bn_recal_janek', 'check-train', '--train-input', found[0], '--bundle', BUNDLE, '--output', BN_CHECK],
       LOG, cwd=CODE, env=ENV, title='bn_recal check-train')
check = json.loads((BN_CHECK/'train_check.json').read_text())
print(f"\nDạng file: {check['kind']}   trạng thái hash manifest: {check['manifest_sha256_status']}")
if check['manifest_sha256_status'] != 'VERIFIED_MATCH':
    print('LƯU Ý: đây là danh sách ID/SHA ảnh, CHƯA xác minh được hash manifest gốc của Tuấn. Ghi điều này vào báo cáo.')
print('Subset:', check['subset']['sources'], 'nguồn, SHA', check['subset'].get('sha256', '')[:16], '...')

## Ô B2 · Ảnh TRAIN Clean của subset + tạo LR2 bằng hàm của Tuấn

- **Ảnh Clean:** lấy từ `cache/textvqa_train_bn1000_<sha>.zip` trên Drive nếu đã có. Nếu chưa, tải lại zip TextVQA 7,07 GB vào thư mục tạm, chỉ trích 1.000 ảnh của subset, kiểm SHA từng ảnh với file TRAIN, rồi lưu bản cache lên Drive.
- **LR2:** tạo bằng `rev4.training_data.native_lr2` (Pillow 11.3.0, bicubic, mỗi chiều còn một nửa, không phóng lại), lưu PNG như cách Tuấn làm với validation.
  - Không có SHA LR2 khóa sẵn cho TRAIN, nên lần đầu ghi SHA vào `train_check/subset_images.csv`.
  - Các lần sau (VM mới), LR2 tạo lại phải **trùng từng byte**, nếu không thì dừng.

In [ ]:
# ===================== Ô B2 · ẢNH TRAIN + LR2 =====================
import csv
subset_csv = BN_CHECK/'subset.csv'
if not subset_csv.is_file():
    ch.fail('Chưa có subset.csv. Chạy Ô B1 trước.')
with subset_csv.open(encoding='utf-8', newline='') as f:
    subset = list(csv.DictReader(f))
items = [(r['clean_relpath'], r['clean_sha256']) for r in subset]
names = [r for r, _ in items]
SUBSET_TAG = ch.sha256(subset_csv)[:12]
TRAIN_CACHE = CACHE/f'textvqa_train_bn1000_{SUBSET_TAG}.zip'
TRAIN_IMAGES, TRAIN_LR2 = T/'textvqa_train_bn', T/'train_lr2_bn'
LOG = LOGS/f'bn_data_{GPU}.log'
if TRAIN_CACHE.is_file():
    ch.extract_members(TRAIN_CACHE, names, TRAIN_IMAGES)
else:
    big = DOWNLOADS/'train_val_images.zip'
    DOWNLOADS.mkdir(parents=True, exist_ok=True)
    if not (big.is_file() and big.stat().st_size == PIN['TEXTVQA_ZIP_BYTES']):
        ch.run(['wget', '-c', '-q', '--show-progress', '-O', big, PIN['TEXTVQA_ZIP_URL']], LOG, title='download TextVQA zip (7 GB)')
    if big.stat().st_size != PIN['TEXTVQA_ZIP_BYTES']:
        ch.fail(f'Zip TextVQA có {big.stat().st_size} byte, kỳ vọng {PIN["TEXTVQA_ZIP_BYTES"]}; chạy lại để tải tiếp.')
    ch.extract_members(big, names, TRAIN_IMAGES)
check_images = ch.verify_images(TRAIN_IMAGES, items)
print('Ảnh Clean TRAIN:', check_images)
if check_images['ok'] != len(items):
    ch.fail(f'Ảnh Clean TRAIN chưa khớp SHA với file TRAIN: {check_images}')
if not TRAIN_CACHE.is_file():
    ch.build_cache_zip(TRAIN_IMAGES, names, TRAIN_CACHE)
    print('Đã lưu cache 1.000 ảnh TRAIN lên Drive:', TRAIN_CACHE)
    big.unlink()
ch.run([PY, '-m', 'mixab.bn_recal_janek', 'prepare-images', '--subset', subset_csv, '--clean-root', TRAIN_IMAGES,
        '--lr2-root', TRAIN_LR2, '--record', BN_CHECK/'subset_images.csv'], LOG, cwd=CODE, env=ENV, title='bn_recal prepare-images')
print('\nÔ B2 xong: 1.000 Clean (SHA khớp) + 1.000 LR2 (native_lr2).')

## Ô B3 · Giai đoạn 0 trên GPU (resume được)

- Chạy lần lượt 5 checkpoint (α = 0, ¼, ½, ¾, 1) của v1, mỗi checkpoint 2.000 ảnh TRAIN, batch 1, FP32. Đường forward giống `rev4.stage_a.forward` (`mode='tensor'`).
- Ô dừng nếu trọng số hoặc buffer bị thay đổi (so SHA toàn bộ `state_dict` trước và sau).
- **Thời gian chưa đo.** Ô in số giây mỗi ảnh sau mỗi 100 ảnh. Ước lượng thô từ A2r: dưới 1 giờ trên A100, vì không chấm 72 setting.
- **Bị ngắt:** tiến độ lưu mỗi 100 ảnh trong `runs/bn_stage0_v1_<GPU>/`. Chạy lại Ô 1 → Ô 4, Ô B1, Ô B2, Ô B3; nó chạy tiếp từ chỗ dừng. Model nào xong rồi sẽ không chạy lại. Đổi code, GPU, subset hay weights thì identity đổi và ô từ chối trộn kết quả.

In [ ]:
# ===================== Ô B3 · GIAI ĐOẠN 0 =====================
pass_flag = T/'doctor_pass.json'
if not pass_flag.is_file():
    ch.fail('Chưa có doctor 0 FAIL trong phiên này. Chạy Ô 4 trước.')
flag = json.loads(pass_flag.read_text())
if flag['gpu'] != GPU or flag['code_zip_sha256'] != code_sha:
    ch.fail('Doctor đã chạy với GPU hoặc code khác. Chạy lại Ô 4.')
if not (BN_CHECK/'subset_images.csv').is_file():
    ch.fail('Chưa có subset_images.csv. Chạy Ô B2 trước.')
BN_OUT = RUNS/f'bn_stage0_v1_{GPU}'
if (BN_OUT/'stage0_summary.json').is_file():
    print(f'Giai đoạn 0 trên {GPU} đã xong trước đó ({BN_OUT}). Không chạy lại; sang Ô B4.')
else:
    log = ch.new_path(LOGS, f'bn_stage0_{GPU}', '.log')
    ch.run([PY, '-m', 'mixab.bn_recal_janek', 'stage0', '--subset-images', BN_CHECK/'subset_images.csv',
            '--train-check', BN_CHECK/'train_check.json', '--clean-root', TRAIN_IMAGES, '--lr2-root', TRAIN_LR2,
            '--model-config', CONFIG_PATH, '--weights-dir', WEIGHTS, '--output', BN_OUT, '--chunk', '100'],
           log, cwd=CODE, env=ENV, title=f'bn_recal stage0 on {GPU}')
    print(f'\nGiai đoạn 0 xong. Kết quả: {BN_OUT}\nLog: {log}')

## Ô B4 · Tổng hợp và đóng gói gửi lại

Các số được in:
- **Lệch giữa thống kê đã lưu và thống kê đo được**, theo từng lớp và từng α:
  - `z_mean`: |Δμ|/σ trung bình theo kênh;
  - `logvar_mean`: |log tỉ lệ phương sai| trung bình;
  - mỗi số tách theo Clean, LR2 và tổng.
- **Phi tuyến:** thống kê đo được của α lệch bao nhiêu so với nội suy tuyến tính thống kê đo được của hai endpoint.
- **Cổng đề xuất (CHƯA DUYỆT):** có tín hiệu nếu `z_mean` của `binarize.1` ở một α giữa lớn hơn 2 lần mức lớn nhất của hai endpoint. Đây chỉ là gợi ý để bàn với Tuấn, **không** tự mở Giai đoạn 1.
- `binarize.4` đo sau `binarize.1` với thống kê đã lưu, nên chỉ để tham khảo.

Cuối ô tạo `export/cs431_bn_stage0_<GPU>_<thời gian>.zip` để gửi lại.

In [ ]:
# ===================== Ô B4 · TỔNG HỢP + ĐÓNG GÓI =====================
BN_OUT = RUNS/f'bn_stage0_v1_{GPU}'
if not (BN_OUT/'stage0_summary.json').is_file():
    ch.fail('Chưa có stage0_summary.json. Chạy Ô B3 trước.')
s = json.loads((BN_OUT/'stage0_summary.json').read_text())
for layer in ('det_head.binarize.1', 'det_head.binarize.4'):
    print(f'\n{layer}  (đã lưu so với đo trên TRAIN)')
    for alpha, row in s['stored_vs_measured'].items():
        r = row[layer]
        print(f"  α={alpha:4}  z_mean all/clean/lr2 = {r['all']['z_mean']:.3f}/{r['clean']['z_mean']:.3f}/{r['lr2']['z_mean']:.3f}"
              f"   logvar_mean all = {r['all']['logvar_mean']:.3f}")
    for alpha, r in s['nonlinearity_vs_endpoint_interp'][layer].items():
        print(f"  phi tuyến α={alpha:4}: z_mean = {r['z_mean']:.3f}, logvar_mean = {r['logvar_mean']:.3f}")
g = s['gate']
print(f"\nCổng ĐỀ XUẤT (chưa duyệt): {g['rule']}\n  mức nền endpoint = {g['endpoint_baseline']:.3f}; "
      f"α giữa = { {k: round(v, 3) for k, v in g['interior'].items()} }\n  tín hiệu: {g['signal']} {g['alphas_with_signal']}")
print('Giai đoạn 1 KHÔNG tự chạy: cần Tuấn duyệt tiêu chí trước.')
for alpha, p in s['profile'].items():
    if p.get('mean_seconds_per_image'):
        print(f"  α={alpha:4}: {p['images']} ảnh, {p['mean_seconds_per_image']:.3f} s/ảnh")

pack = ch.new_path(EXPORT, f'cs431_bn_stage0_{GPU}', '.zip')
with zipfile.ZipFile(pack, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(BN_OUT.iterdir()):
        if p.is_file() and not p.name.startswith('.'):
            zf.write(p, f'stage0/{p.name}')
    for name in ('train_check.json', 'subset.csv', 'subset_images.csv'):
        if (BN_CHECK/name).is_file():
            zf.write(BN_CHECK/name, f'train_check/{name}')
    for folder, pattern in ((REPORTS, f'*_{GPU}_*.json'), (LOGS, f'*bn*{GPU}*.log'), (LOGS, f'doctor_{GPU}.log')):
        for p in sorted(folder.glob(pattern)):
            zf.write(p, f'{folder.name}/{p.name}')
    zf.writestr('code_zip_sha256.txt', code_sha + '\n')
print(f'\nĐã đóng gói: {pack} ({pack.stat().st_size/1024:.0f} KiB). Có thể tải trực tiếp:')
try:
    from google.colab import files
    files.download(str(pack))
except Exception as exc:
    print('Không tự tải được, hãy tải từ Drive:', exc)

## Giai đoạn 1: chưa mở

Giai đoạn 1 tính lại thống kê 2 lớp BN `binarize` trên cùng subset TRAIN rồi chấm counts 72 setting, có đối chứng trước/sau và endpoint. Nó **chỉ được thêm vào notebook** khi đủ cả ba điều kiện:
1. Tuấn duyệt cổng Giai đoạn 0, ngưỡng "đáng xác nhận" (§6–§7 của thiết kế), và cách xác nhận (§8).
2. Giai đoạn 0 có tín hiệu theo cổng đã duyệt.
3. Kết quả Giai đoạn 0 đã được đọc và ghi vào WORKLOG.

Không mở TEST. Kết quả trên A/B chỉ là khám phá.